# Módulo 4 · Sistemas monoagente

**Duración sugerida:** 4 horas · **Entrega:** agente con ≥2 herramientas, registro
de ejecución y pruebas de casos exitosos y fallidos.

## Objetivos del laboratorio

- Entender el **ciclo de razonamiento**: observar → seleccionar acción → ejecutar → revisar.
- Definir **herramientas (tools)** y que el agente las **seleccione**.
- Registrar la **ejecución** y comprobar si la respuesta está **respaldada** por resultados.
- Probar **casos exitosos y fallidos**.

## Conexión con el proyecto ARAS

El **agente coordinador** de ARAS decide qué herramienta usar (búsqueda en
documentos, consulta de datos, cálculo) para responder una pregunta de investigación.

## 1. Dependencias

In [ ]:
%pip install -q langchain-core langchain-ollama langgraph pydantic

## 2. Diferencias LLM / pipeline / agente

- **LLM:** responde con lo que "sabe" (puede alucinar).
- **Pipeline:** pasos fijos, sin decisiones en tiempo de ejecución.
- **Agente:** decide **qué herramienta** usar y **cuándo parar**, en un ciclo.

## 3. Definición de herramientas

Una *tool* es una función con nombre, descripción y esquema de argumentos. El
agente usa la descripción para decidir cuál invocar.

In [ ]:
from langchain_core.tools import tool

@tool
def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática simple y devuelve el resultado numérico."""
    permitidos = set("0123456789+-*/(). ")
    if any(c not in permitidos for c in expresion):
        return "Error: la expresión contiene caracteres no permitidos."
    try:
        return str(eval(expresion, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error al evaluar: {e}"

@tool
def buscar_en_documentos(consulta: str) -> str:
    """Busca en una colección local de documentos académicos y devuelve fragmentos relevantes."""
    documentos = {
        "evaluacion": "La evaluación de agentes mide relevancia, uso correcto de herramientas y trazabilidad.",
        "mcp": "Model Context Protocol estandariza el acceso a herramientas y recursos externos.",
        "rag": "RAG recupera fragmentos relevantes antes de generar la respuesta final.",
    }
    resultados = [f"- {v}" for k, v in documentos.items() if consulta.lower() in v.lower() or k in consulta.lower()]
    return "\n".join(resultados) if resultados else "No se encontraron documentos relevantes."

@tool
def consultar_datos(indicador: str) -> str:
    """Consulta un conjunto de datos estructurados y devuelve el valor del indicador."""
    datos = {
        "articulos_2024": 128,
        "articulos_2023": 95,
        "precision_promedio": 0.82,
    }
    clave = indicador.lower().replace(" ", "_")
    return str(datos.get(clave, f"Indicador '{indicador}' no encontrado."))

herramientas = [calculadora, buscar_en_documentos, consultar_datos]
print("Herramientas registradas:", [h.name for h in herramientas])

## 4. Ciclo de razonamiento manual

Antes de usar un framework, ilustramos el ciclo a mano: **observar** (mensaje),
**seleccionar** (elegir tool), **ejecutar**, **revisar** (¿responde ya?).

In [ ]:
def ciclo_manual(pregunta):
    registro = []
    # 1. Observar: interpretamos la intención.
    registro.append(f"OBSERVAR: {pregunta}")
    # 2. Seleccionar acción según la intención (aquí, reglas simples).
    if any(c.isdigit() for c in pregunta) and any(op in pregunta for op in "+-*/"):
        accion = "calculadora"
        argumento = pregunta.split(":")[-1].strip()
    elif "documento" in pregunta.lower() or "artículo" in pregunta.lower():
        accion = "buscar_en_documentos"
        argumento = pregunta
    else:
        accion = "consultar_datos"
        argumento = pregunta
    registro.append(f"SELECCIONAR: {accion}({argumento!r})")
    # 3. Ejecutar.
    resultado = {h.name: h for h in herramientas}[accion].invoke(argumento)
    registro.append(f"EJECUTAR: {resultado}")
    # 4. Revisar: si hay resultado, terminar.
    registro.append("REVISAR: resultado disponible -> fin")
    return resultado, registro

resp, log = ciclo_manual("calcula: 12 * 7")
for linea in log:
    print(linea)
print("\nRespuesta:", resp)

## 5. Agente con LangGraph (`create_react_agent`)

LangGraph ofrece el bucle ReAct ya implementado: el modelo decide qué tool llamar
y cuándo terminar, con control del número de iteraciones.

In [ ]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

llm = ChatOllama(model="llama3.2", temperature=0)
agente = create_react_agent(llm, herramientas)

def ejecutar(pregunta):
    resultado = agente.invoke({"messages": [("user", pregunta)]})
    print("=== REGISTRO DE EJECUCIÓN (mensajes) ===")
    for m in resultado["messages"]:
        print(f"[{type(m).__name__}] {m.content}")
    return resultado["messages"][-1].content

print("\n=== CASO 1: cálculo ===")
ejecutar("¿Cuánto es 15 * 8?")

In [ ]:
print("\n=== CASO 2: búsqueda en documentos ===")
ejecutar("Busca documentos sobre evaluación de agentes.")

In [ ]:
print("\n=== CASO 3: consulta de datos ===")
ejecutar("Dame el valor del indicador artículos 2024.")

## 6. Casos fallidos y límites de iteración

Los agentes pueden fallar: no encontrar herramienta, bucles infinitos o
respuestas no respaldadas. Configuramos **límites de iteración** y observamos.

In [ ]:
# Límite de iteraciones para evitar bucles.
agente_limitado = create_react_agent(llm, herramientas)

print("=== CASO FALLIDO: petición fuera del alcance de las herramientas ===")
try:
    resultado = agente_limitado.invoke(
        {"messages": [("user", "Reserva una mesa en un restaurante de Madrid.")]},
        config={"recursion_limit": 5},
    )
    print(resultado["messages"][-1].content)
except Exception as e:
    print("El agente no pudo resolver la tarea (fuera de alcance):", e)

## 7. Comprobar respaldo de la respuesta

El registro permite **auditar** si la respuesta final se apoya en resultados de
herramientas y no en una alucinación.

In [ ]:
def esta_respaldada(resultado):
    mensajes = resultado["messages"]
    herramientas_llamadas = [m for m in mensajes if getattr(m, "tool_calls", None)]
    tiene_resultado_tool = any(getattr(m, "role", "") == "tool" for m in mensajes)
    return bool(herramientas_llamadas) and tiene_resultado_tool

resultado = agente.invoke({"messages": [("user", "¿Cuánto es 3 + 4?")]})
print("Respuesta respaldada por herramientas:", esta_respaldada(resultado))

## Resumen del laboratorio

- Definimos **3 herramientas** (calculadora, búsqueda, datos).
- Ilustramos el **ciclo de razonamiento** manual y con `create_react_agent`.
- Registramos la **ejecución** (qué herramienta se llamó y su resultado).
- Probamos **casos exitosos y fallidos**, con **límites de iteración** y
  verificación de **respaldo**.

**Entregable del Módulo 4:** agente individual con ≥2 herramientas, registro de
ejecución y pruebas de casos exitosos y fallidos.